In [0]:
%pip install --quiet gspread google-auth
dbutils.library.restartPython()

In [0]:
import json
import gspread
import pandas as pd
from datetime import datetime, timezone
from google.oauth2.service_account import Credentials

SHEET_ID = "1bam2ByYUsriJbkfJp5o3SQXRBtwOWSNNFQ1lCziT1yc"
TAB = "_connection_test"

# Sign in as the service account using the secret
sa_info = json.loads(dbutils.secrets.get("power-prices", "gcp-sa-json"))
creds = Credentials.from_service_account_info(
    sa_info, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)
sh = gc.open_by_key(SHEET_ID)
print("Opened sheet:", sh.title)

# Get or create the test tab
try:
    ws = sh.worksheet(TAB)
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=TAB, rows=20, cols=5)

# Small test table
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
df = pd.DataFrame({
    "zone": ["DE_LU", "FR", "ES", "NL"],
    "test_price_eur_mwh": [85.20, 72.10, 64.50, 88.90],
    "written_at": [now] * 4,
})

ws.clear()
ws.update([df.columns.tolist()] + df.astype(str).values.tolist())

# Read back to confirm
rows = ws.get_all_records()
print(f"Wrote and read back {len(rows)} rows to tab '{TAB}' at {now}")
print(rows[0])